In [1]:
import duckdb
import requests
import numpy as np
import json

# ── KONFIGURATION ────────────────────────────────────────────────────────────
# Ersätt med din nya dators faktiska IP-adress om du kör från den gamla datorn.
# Om du testar lokalt på samma dator som servern, använd 'localhost'.
SERVER_IP = "127.0.0.1"  
SERVER_URL = f"http://{SERVER_IP}:8080/api/analyze"
DB_PATH = "../capillary.db"


print(f"Ansluter till databasen: {DB_PATH}...")
con = duckdb.connect(DB_PATH)

query = """
SELECT row_id, age, gender, value, albumin, antitrypsin, orosomukoid, 
        haptoglobin, crp, igg, iga, igm, s_kappa, s_lambda, s_kl_kvot, 
        u_albumin_krea, u_igg_krea, u_kappa_krea, u_lambda_krea, u_hc_krea, 
        u_kappa, u_lambda, u_kreatinin
FROM protein_data WHERE observation_nr = 1
"""

# Hämta som en Pandas DataFrame
df = con.execute(query).df()
con.close()

if df.empty:
    print("Kunde inte hitta något fall i databasen som matchade sökkriterierna!")


df = df.replace({np.nan: None})

Ansluter till databasen: ../capillary.db...


In [2]:

# ── FORMATTERA TILL DICT / JSON ──────────────────────────────────────────
# .iloc[0].to_dict() konverterar raden till en ren Python-dict.
# Det magiska här är att kurvan (som lagras som en array/lista i DuckDB) 
# automatiskt blir till en helt vanlig Python-lista [120, 115, ...].
# SQL NULL blir automatiskt till None (vilket blir 'null' i JSON).

patient_data = df.iloc[1].to_dict()

# Om 'value' är en NumPy-array, konvertera den till en ren Python-lista
if isinstance(patient_data.get('value'), np.ndarray):
    patient_data['value'] = patient_data['value'].tolist()
print(f"\nHittade fall ID: {patient_data.get('id')}. Förbereder JSON-anrop...")

# ── SKICKA TILL SERVER ───────────────────────────────────────────────────
print(f"Skickar POST-anrop till: {SERVER_URL}...")
try:
    # requests.post med parametern 'json=' konverterar automatiskt vår 
    # dict till en JSON-sträng och sätter rätt headers (application/json)
    response = requests.post(SERVER_URL, json=patient_data, timeout=10)

    req = response.request
    print("=== REQUEST ===")
    print(f"Method:  {req.method}")
    print(f"URL:     {req.url}")
    print(f"Headers: {dict(req.headers)}")
    print(json.dumps(json.loads(req.body), indent=2, ensure_ascii=False))
    print("===============")
    
    # Kontrollera om servern svarade med en felkod (t.ex. 400 eller 500)
    response.raise_for_status()
    
    # Dekoda JSON-svaret från Bottle-servern
    result = response.json()
    
    # ── SKRIV UT SVARET snyggt ───────────────────────────────────────────
    print("\n" + "="*60)
    print(f" SVALT RESPONS FRÅN SERVER (Status: {result.get('status')})")
    print("="*60)
    print(f"Patient ID: {result.get('id')}")
    print("\nModell-sannolikheter:")
    for model, prob in result.get("predictions", {}).items():
        if "probability" in model:
            print(f"  - {model}: {prob*100:.2f}%")
        else:
            print(f"  - {model}: {prob}")
            
    print(f"\nBeställda tilläggsanalyser: {result.get('bestalda_analyser')}")
    print("\nGenererat maskinutlåtande:")
    print(f"\"{result.get('utlatande')}\"")
    print("="*60 + "\n")
    
except requests.exceptions.HTTPError as http_err:
    print(f"\n[HTTP FEL]: Servern svarade med felkod: {response.status_code}")
    try:
        # Försök skriva ut serverns interna felmeddelande om det finns
        server_error = response.json()
        print(f"Meddelande från servern: {server_error.get('message')}")
    except Exception:
        print(f"Råtext från servern: {response.text}")
        
except requests.exceptions.ConnectionError:
    print(f"\n[ANSLUTNINGSFEL]: Kunde inte nå servern på {SERVER_URL}.")
    print("Kontrollera att server.py körs och att du har angett rätt IP-adress!")
    



Hittade fall ID: None. Förbereder JSON-anrop...
Skickar POST-anrop till: http://127.0.0.1:8080/api/analyze...
=== REQUEST ===
Method:  POST
URL:     http://127.0.0.1:8080/api/analyze
Headers: {'User-Agent': 'python-requests/2.34.2', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive', 'Content-Length': '1698', 'Content-Type': 'application/json'}
{
  "row_id": 3,
  "age": 61.25,
  "gender": "K",
  "value": [
    1,
    1,
    1,
    1,
    1,
    2,
    2,
    2,
    1,
    1,
    2,
    1,
    1,
    1,
    1,
    2,
    3,
    3,
    5,
    6,
    6,
    6,
    7,
    7,
    8,
    8,
    8,
    9,
    10,
    10,
    11,
    11,
    12,
    16,
    18,
    20,
    24,
    28,
    32,
    33,
    32,
    27,
    23,
    19,
    15,
    13,
    12,
    11,
    11,
    12,
    14,
    16,
    18,
    20,
    23,
    24,
    26,
    29,
    31,
    32,
    33,
    34,
    35,
    36,
    38,
    43,
    50,
    65,
    87,
    115,
    145,
    179,
    215,
